# 49. 원판·자세 모델을 Mac GPU로 실행하면 얼마나 빨라질까?

PyTorch에서 장치만 CPU에서 MPS로 바꾼 뒤, 기존 배포 후보인 ONNX Runtime CPU도 같은 조건에서 잽니다. 대상은 검증 c04 116장, 입력은 고정 FP32 640×384, 배치 1입니다. 동일한 입력을 두 모델에 한 번만 준비합니다. 처리 순서를 뒤집어 조건별로 2회 측정하고, 매 회차 10프레임을 예열합니다.

측정 범위는 한 장의 디코딩부터 두 모델의 NMS와 원본 좌표 복원까지입니다. MPS는 입력 이동·GPU 완료 대기·CPU 결과 반환을 포함합니다. 시간 연결·각도 집계·오버레이·인코딩·파일 저장과 모델 초기화는 측정 밖입니다. GPU가 있는 Mac의 프로젝트 커널에서 실행하세요.

In [ ]:
from pathlib import Path
from datetime import datetime
import sys

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "wlcv/config.py").is_file())
sys.path.insert(0, str(root))
from wlcv.evaluation.mac_mps import benchmark
import matplotlib.pyplot as plt

out = root / "outputs"
plate = out / "onnx_plate_parity/20261125-194508-519898/plate.onnx"
pose = out / "onnx_pose_parity/20261125-200823-826057/pose.onnx"
folder = out / "mac_mps_latency" / datetime.now().strftime("%Y%m%d-%H%M%S-%f")
folder.mkdir(parents=True)
frames, domain = benchmark(plate, pose, folder)
print(folder)

## 같은 프레임의 처리 시간과 출력

p50은 지연 중앙값, p95는 느린 프레임까지 볼 수 있는 95백분위입니다. 최초 PyTorch CPU 실행의 프레임별 후보 수·표시 상태·좌표·점수와 대조합니다. 후보 수가 달라 좌표 대조가 불가능하면 그 장은 빈칸으로 남습니다. 일치하지 않은 장수와 비교 가능한 장수를 함께 확인하세요.

In [ ]:
latency = frames.groupby("runtime").total_ms.quantile([.50, .95]).unstack()
latency.columns = ["p50_ms", "p95_ms"]
display(latency.round(2))
display(frames.groupby(["trial", "runtime"]).total_ms.quantile([.50, .95]).unstack().round(2))
checks = frames.groupby("runtime").agg(rows=("frame","size"), compared=("coordinate_delta_px","count"),
    all_states_equal=("same_counts_and_states","all"), max_coord_px=("coordinate_delta_px","max"),
    max_score_diff=("score_delta","max"))
display(checks)
display(frames.loc[~frames.same_counts_and_states, ["runtime","frame"]].drop_duplicates())
display(domain.round(6))
latency.to_csv(folder / "latency.csv")
checks.to_csv(folder / "checks.csv")

In [ ]:
ax = latency.plot.bar(rot=0, figsize=(6, 3.5))
ax.axhline(1000/30, ls="--", color="gray", label="30 fps frame interval")
ax.set(xlabel="", ylabel="Decode to coordinates (ms)")
ax.legend()
ax.figure.tight_layout()
ax.figure.savefig(folder / "runtime_latency.png", dpi=140)
plt.show()